In [45]:
import os
from pathlib import Path
import pprint
import numpy as np

import probeinterface as prb
import spikeinterface.extractors as se
import spikeinterface.preprocessing as spre
import spikeinterface.sorters as ss
from spikeinterface.metrics import ComputeTemplateMetrics
import spikeinterface.exporters as sepx
import spikeinterface.core as sc
from spikeinterface.curation import bombcell_label_units, bombcell_get_default_thresholds


In [46]:
base_path = Path('/Users/jeremy/Desktop/spikeinterface-results/')


In [47]:
# the results can be read back for futur session
#rec = se.read_openephys(folder_path = '/Users/jeremy/Desktop/gpr26_161_ephys/20260914/run1_resting_tip_ref/Record Node 101/experiment1/recording1/',stream_name = 'Record Node 101#Neuropix-PXI-112.ProbeA')
#rec = se.read_openephys(folder_path = '/Users/jeremy/Desktop/gpr26_161_ephys/20260914/run2_resting_gnd_ref/Record Node 101/experiment1/recording1/',stream_name = 'Record Node 101#Neuropix-PXI-112.ProbeA')
rec = se.read_openephys(folder_path = '/Users/jeremy/Desktop/gpr26_161_ephys/20260914/run3/Record Node 101/experiment1/recording1/',stream_name = 'Record Node 101#Neuropix-PXI-112.ProbeA')

In [48]:
rec

OpenEphysBinaryRecordingExtractor: 384 channels - 30.0kHz - 1 segments - 22,212,320 samples 
                                   740.41s (12.34 minutes) - int16 dtype - 15.89 GiB

In [ ]:
sorting = sc.load(base_path / 'run3/kilosort4/')


ValueError: /users/jmurph14/data/jmurph14/ephys/gpr26_161_ephys/20260914/run3/Record Node 101 is not a valid Open Ephys binary folder. No 'structure.oebin' files were found in sub-folders.

In [37]:
sortA = sc.create_sorting_analyzer(sorting,rec,sparse=True,format='memory')

In [38]:
compute_dict = {
    'random_spikes': {},
    'noise_levels':{},
    'waveforms':{},
    'principal_components': {'n_components': 3, 'mode': 'by_channel_local'},
    'templates': {},
    'template_similarity':{},
    'spike_locations':{},
    'spike_amplitudes':{},
    'correlograms':{},
}
sortA.compute(compute_dict)

noise_level (no parallelization):   0%|          | 0/20 [00:00<?, ?it/s]

compute_waveforms (no parallelization):   0%|          | 0/490 [00:00<?, ?it/s]

Fitting PCA:   0%|          | 0/226 [00:00<?, ?it/s]

/Users/jeremy/Documents/python_environments/spikeInterface/lib/python3.12/site-packages/sklearn/decomposition/_incremental_pca.py:362: RuntimeWarning: invalid value encountered in divide
  explained_variance_ratio = S**2 / np.sum(col_var * n_total_samples)
/Users/jeremy/Documents/python_environments/spikeInterface/lib/python3.12/site-packages/sklearn/decomposition/_incremental_pca.py:362: RuntimeWarning: divide by zero encountered in divide
  explained_variance_ratio = S**2 / np.sum(col_var * n_total_samples)
/Users/jeremy/Documents/python_environments/spikeInterface/lib/python3.12/site-packages/sklearn/decomposition/_incremental_pca.py:362: RuntimeWarning: divide by zero encountered in divide
  explained_variance_ratio = S**2 / np.sum(col_var * n_total_samples)
/Users/jeremy/Documents/python_environments/spikeInterface/lib/python3.12/site-packages/sklearn/decomposition/_incremental_pca.py:362: RuntimeWarning: divide by zero encountered in divide
  explained_variance_ratio = S**2 / np.

Projecting waveforms:   0%|          | 0/226 [00:00<?, ?it/s]

Compute : spike_locations + spike_amplitudes (no parallelization):   0%|          | 0/490 [00:00<?, ?it/s]

In [39]:
sortA.compute('quality_metrics')
sortA.compute('template_metrics')

/Users/jeremy/Documents/python_environments/spikeInterface/lib/python3.12/site-packages/spikeinterface/metrics/quality/misc_metrics.py:182: RuntimeWarning: divide by zero encountered in scalar divide
  snrs[unit_id] = np.abs(amplitude) / noise
/Users/jeremy/Documents/python_environments/spikeInterface/lib/python3.12/site-packages/numpy/_core/_methods.py:219: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/Users/jeremy/Documents/python_environments/spikeInterface/lib/python3.12/site-packages/numpy/_core/_methods.py:178: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
/Users/jeremy/Documents/python_environments/spikeInterface/lib/python3.12/site-packages/numpy/_core/_methods.py:211: RuntimeWarning: invalid value encountered in divide
  ret = ret.dtype.type(ret / rcount)
/Users/jeremy/Documents/python_environments/spikeInterface/lib/python3.12/site-packages/spikeinterfa

In [40]:
sortA.get_loaded_extension_names()

['random_spikes',
 'noise_levels',
 'waveforms',
 'principal_components',
 'templates',
 'template_similarity',
 'correlograms',
 'spike_locations',
 'spike_amplitudes',
 'quality_metrics',
 'template_metrics']

In [41]:
sortA.save_as(folder=base_path / 'run1/sorting_analyzer/',format='binary_folder')


/Users/jeremy/Documents/python_environments/spikeInterface/lib/python3.12/site-packages/spikeinterface/core/core_tools.py:968: UserWarning: The extractor is not serializable to file. The provenance will not be saved.
  warnings.warn("The extractor is not serializable to file. The provenance will not be saved.")
/Users/jeremy/Documents/python_environments/spikeInterface/lib/python3.12/site-packages/spikeinterface/core/basesorting.py:646: UserWarning: The recording registered to this sorting object will not be saved to disk. Reloading the sorting later will not include the recording
  warnings.warn(


SortingAnalyzer: 384 channels - 226 units - 1 segments - binary_folder - sparse - has recording
Loaded 11 extensions: random_spikes, noise_levels, waveforms, principal_components, templates, template_similarity, correlograms, spike_locations, spike_amplitudes, quality_metrics, template_metrics

In [42]:
# auto-label units with Bombcell
labels = bombcell_label_units(
    sorting_analyzer=sortA,
    thresholds=bombcell_get_default_thresholds()
)

In [43]:
labels = labels.to_numpy()

In [44]:
np.save(base_path / 'run1/bombcell_label.npy',labels,allow_pickle=True)

In [ ]:

from spikeinterface.widgets import plot_sorting_summary

# this will open the GUI in a different window
plot_sorting_summary(sorting_analyzer=sortA, curation=True, backend='spikeinterface_gui')

: 

In [ ]:
sortA.get


['random_spikes',
 'noise_levels',
 'waveforms',
 'principal_components',
 'templates',
 'template_similarity',
 'correlograms',
 'spike_locations',
 'spike_amplitudes',
 'quality_metrics',
 'template_metrics',
 'unit_locations']

In [25]:
ext = sortA.get_extension("spike_locations")
ext_data = ext.get_data()


In [26]:
ext_data

array([(19.947691 ,  342.32257), (13.52053  , 2828.929  ),
       ( 9.8930025,  823.7385 ), ..., (14.556622 , 2533.9763 ),
       (17.422897 ,  448.3112 ), (17.068161 ,  494.36267)],
      shape=(403543,), dtype=[('x', '<f4'), ('y', '<f4')])

In [32]:
ext = sortA.get_extension('template_metrics')
ext_data = ext.get_data()
ext_data

,peak_to_trough_duration,trough_half_width,peak_half_width,repolarization_slope,recovery_slope,num_positive_peaks,num_negative_peaks,main_to_next_extremum_duration,peak_before_to_trough_ratio,peak_after_to_trough_ratio,peak_before_to_peak_after_ratio,main_peak_to_trough_ratio,trough_width,peak_before_width,peak_after_width,waveform_baseline_flatness,velocity_above,velocity_below,exp_decay,spread
0,0.000347,NaN,NaN,NaN,-1.426168e+05,0,0,0.001077,1.095097,1.072903,1.020686,1.095097,NaN,NaN,NaN,0.139393,NaN,775.100974,0.042788,NaN
1,0.002823,NaN,NaN,NaN,9.667154e+05,0,0,0.002823,0.071703,0.878688,0.081602,0.878688,NaN,NaN,NaN,0.137107,NaN,NaN,0.050410,NaN
2,0.001917,NaN,NaN,NaN,-4.527467e+04,0,0,0.001917,1.111279,1.122977,0.989583,1.122977,NaN,NaN,NaN,0.139158,NaN,NaN,NaN,NaN
3,0.002770,NaN,NaN,NaN,9.295778e+05,0,0,0.002770,0.948945,0.880842,1.077316,0.948945,NaN,NaN,NaN,0.146038,NaN,NaN,NaN,NaN
4,0.000150,NaN,NaN,NaN,-4.951536e+05,0,0,0.002650,1.123554,1.097909,1.023358,1.123554,NaN,NaN,NaN,0.149410,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
221,-0.002913,NaN,NaN,NaN,NaN,0,0,0.002387,1.147131,0.079306,14.464553,1.147131,NaN,NaN,NaN,0.162415,NaN,NaN,NaN,NaN
222,0.002823,NaN,NaN,NaN,2.543997e+06,0,0,0.002823,0.073588,0.906171,0.081207,0.906171,NaN,NaN,NaN,0.133073,NaN,NaN,NaN,NaN
223,-0.002913,NaN,NaN,NaN,NaN,0,0,0.000027,1.110658,0.081815,13.575214,1.110658,NaN,NaN,NaN,0.148327,NaN,NaN,NaN,NaN
224,-0.002913,NaN,NaN,NaN,NaN,0,0,0.000450,1.110663,0.079408,13.986749,1.110663,NaN,NaN,NaN,0.104786,NaN,NaN,NaN,NaN
